## 1) Làm sạch và chuẩn hoá dữ liệu bất động sản TP.HCM
Làm sạch & Chuẩn hoá dữ liệu bất động sản TP.HCM

Xử lý ngoại lai (IQR), điền giá trị thiếu, chuẩn hoá tỷ lệ & địa chỉ.

Kết quả: dữ liệu sạch sẵn sàng cho huấn luyện mô hình.

## 2)Import thư viện

In [2]:
# 🧭 Cấu hình và import

import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# File đầu vào / đầu ra
input_file = "moso_api_data.csv"
out_csv = "moso_clean.csv"
out_report = "moso_clean_report.txt"

# Tham số xử lý
iqr_k = 1.5     # hệ số IQR
impute_method = "median"   # "none" | "median" | "mean"


## 3)Hàm chuyển chuỗi → số

In [3]:
def to_float(x):
    s = str(x).strip() if x is not None else ""
    if s == "" or s.lower() in {"nan", "null", "none"}:
        return np.nan
    s = re.sub(r"(m2|m²|㎡)", "", s, flags=re.I).replace(" ", "")
    if re.search(r"\d+,\d+$", s):
        s = s.replace(",", ".")
    s = re.sub(r"(?<=\d)[\.,](?=\d{3}(\D|$))", "", s)
    try:
        return float(s)
    except:
        return np.nan

## 4)Hàm parse giá sang VND

In [4]:
def parse_price_to_vnd(v):
    s = str(v).lower().strip() if v is not None else ""
    if s == "" or "thỏa thuận" in s or "thoa thuan" in s:
        return np.nan
    m = re.search(r"(\d+(?:[.,]\d+)?)\s*(t[yỷ]|tỷ|ty)", s)
    if m:
        n = float(m.group(1).replace(",", "."))
        val = n * 1_000_000_000
        tail = re.search(r"ty\s*([0-9]{2,3})\b", s)
        if tail:
            val += float(tail.group(1)) * 1_000_000
        return val
    m = re.search(r"(\d+(?:[.,]\d+)?)\s*(triệu|tr|million)", s)
    if m:
        return float(m.group(1).replace(",", ".")) * 1_000_000
    n = to_float(s)
    if n and n <= 200 and "đ" not in s and "vnd" not in s:
        return n * 1_000_000_000
    return n or np.nan

## 5) Hàm tính tuổi bài đăng

In [5]:
def parse_age_days(x):
    s = str(x).strip() if x is not None else ""
    if s == "":
        return np.nan
    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    if pd.isna(dt):
        return np.nan
    return (pd.Timestamp.today().normalize() - pd.Timestamp(dt)).days

## 6) IQR (lọc ngoại lai) + scale(chuẩn hoá thang đo) + fnum (định dạng số)

In [6]:
def iqr_mask(series, k=3):
    s = pd.to_numeric(series, errors="coerce")
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if not np.isfinite(iqr) or iqr == 0:
        return s, (np.nan, np.nan), 0
    lo, hi = q1 - k * iqr, q3 + k * iqr
    before = s.isna().sum()
    s = s.mask((s < lo) | (s > hi))
    after = s.isna().sum()
    return s, (lo, hi), int(after - before)

def scale_keep_nan(col, scaler_cls):
    s = pd.to_numeric(col, errors="coerce")
    mask = s.notna()
    out = pd.Series(np.nan, index=s.index)
    if mask.any():
        sc = scaler_cls().fit(s[mask].to_numpy().reshape(-1, 1))
        out.loc[mask] = sc.transform(s[mask].to_numpy().reshape(-1, 1)).ravel()
    return out

def fnum(x):
    return "nan" if pd.isna(x) else f"{float(x):.1f}"


## 7) Hàm xử lý địa chỉ

In [7]:

HCM_PAT = re.compile(r"(tp\.?\s*hồ\s*chí\s*minh|thành\s*phố\s*hồ\s*chí\s*minh|hcmc?|sài\s*gòn)", re.I)
_SKIP_DIST = re.compile(r"(?i)\b(Ấp|Ap|Thôn|Thon|Tổ|To|KDC|Khu|Đường|Duong|Quốc\s*lộ|QL|Hẻm|Hem|Ngõ|Ngo|Số)\b")
RURAL = {"Bình Chánh","Nhà Bè","Hóc Môn","Củ Chi","Cần Giờ"}
NAMED_DIST = {"Tân Phú","Gò Vấp","Bình Thạnh","Phú Nhuận","Bình Tân","Tân Bình"}

def _clean(s): return re.sub(r"\s+", " ", str(s).strip())
def _is_hcm_token(t): return bool(HCM_PAT.search(t))

# số “mềm” 1–2 chữ số (chịu dấu chấm/dấu cách, v.v.)
_NUM_SOFT = re.compile(r"^\s*[^0-9]*?(\d{1,2})[^0-9]*?\s*$")
def get_num_token(s):
    if s is None: return None
    m = _NUM_SOFT.match(str(s))
    try:
        return int(m.group(1)) if m else None
    except:
        return None

def norm_quan_token(tok):
    t = _clean(tok)
    if re.search(r"(?i)thủ\s*đức", t): return "TP Thủ Đức"
    if re.match(r"(?i)h(uyện|\.)\s*", t):
        name = _clean(re.sub(r"(?i)h(uyện|\.)\s*", "", t))
        return "Huyện " + name if name else ""
    if re.match(r"(?i)q(uận|\.)\s*\d{1,2}\b", t):
        n = re.search(r"(\d{1,2})", t).group(1)
        return f"Quận {int(n)}"
    if t in RURAL: return "Huyện " + t
    if t in NAMED_DIST: return "Quận " + t
    return ""  # không phải quận/huyện

def norm_phuong_token(tok, parent_d):
    t = _clean(tok)
    if re.match(r"(?i)tt\.|thị\s*trấn", t):
        name = re.sub(r"(?i)tt\.|thị\s*trấn", "", t).strip(" ,.-")
        return "Thị trấn " + name if name else "Thị trấn"
    if re.match(r"(?i)p\.|p\s+|phường\s+", t):
        name = re.sub(r"(?i)p\.|p\s+|phường\s+", "", t).strip(" ,.-")
        return "Phường " + name if name else "Phường"
    if re.match(r"(?i)x\.|xã\s+", t):
        name = re.sub(r"(?i)x\.|xã\s+", "", t).strip(" ,.-")
        return "Xã " + name if name else "Xã"
    # phường số trần (liền trước quận/TP Thủ Đức)
    n = get_num_token(t)
    if n and 1 <= n <= 30 and (parent_d.startswith("Quận") or parent_d.startswith("TP Thủ Đức")):
        return f"Phường {n}"
    # tên chữ không tiền tố -> suy theo cấp trên (né token đường/ấp)
    if not _SKIP_DIST.search(t) and not norm_quan_token(t):
        return ("Xã " if parent_d.startswith("Huyện") else "Phường ") + t
    return ""

def split_hcm(addr):
    norm = _clean(addr)
    if not norm: return ("Hồ Chí Minh", "", "", "", "")

    toks = [t.strip() for t in norm.split(",") if t.strip()]
    toks = [t for t in toks if not _is_hcm_token(t) and not re.search(r"(?i)\btỉnh|thành\s*phố|tp\.", t)]
    tinh, quan, phuong, duong = "Hồ Chí Minh", "", "", ""
    if not toks: return (tinh, quan, phuong, duong, norm)

    # 1) tìm quận/huyện từ phải → trái (quận chữ/Thủ Đức)
    dist_i = None
    for i in range(len(toks)-1, -1, -1):
        d = norm_quan_token(toks[i])
        if d:
            quan, dist_i = d, i
            break

    # 1b) nếu chưa thấy: tìm số 1–12 từ phải → trái (không chỉ token cuối)
    if not quan:
        for i in range(len(toks)-1, -1, -1):
            n = get_num_token(toks[i])
            if n and 1 <= n <= 12:
                # nếu bên trái là từ “đường/ấp/QL…” thì bỏ qua
                left = toks[i-1] if i-1 >= 0 else ""
                if not _SKIP_DIST.search(left):
                    quan, dist_i = f"Quận {n}", i
                    break

    # 2) phường/xã: token liền trước quận (nếu có)
    ward_i = None
    if dist_i is not None and dist_i-1 >= 0:
        w = norm_phuong_token(toks[dist_i-1], quan)
        if w:
            phuong, ward_i = w, dist_i-1

    # xóa đã dùng (xóa theo thứ tự giảm dần)
    for idx in sorted([i for i in [ward_i, dist_i] if i is not None], reverse=True):
        toks.pop(idx)

    # 3) phần còn lại là đường/số nhà
    duong = ", ".join(toks) if toks else ""
    return (tinh, quan, phuong, duong, norm)


## 8) Load dữ liệu

In [8]:
df = pd.read_csv(input_file)
print(f"📂 Đã đọc {len(df)} dòng từ {input_file}")
print("Xem 5 dòng đầu:")
display(df.head(5))


📂 Đã đọc 2230 dòng từ moso_api_data.csv
Xem 5 dòng đầu:


,URL,Giá,Diện tích sử dụng,Diện tích đất,Phòng ngủ,Phòng tắm,Giấy tờ pháp lý,Ngày đăng,Địa chỉ
0,https://moso.vn/ban-690388d2d638883988715f20,23.5 Tỷ,256.51,162.0,4.0,4.0,certificate,30/10/2025,"46 Đường Số 20, Hiệp Bình Chánh, Thủ Đức, Hồ C..."
1,https://moso.vn/ban-690375a5d638883988712dd5,4.8 Tỷ,NaN,79.8,NaN,NaN,certificate,30/10/2025,"Đường Lê Đức Thọ, 16, Gò Vấp, Hồ Chí Minh"
2,https://moso.vn/ban-69034b76d63888398870c2cf,4.8 Tỷ,60.60,77.1,6.0,6.0,certificate,30/10/2025,"2/23A Đường số 13, Linh Xuân, Thủ Đức, Hồ Chí ..."
3,https://moso.vn/ban-69034961d63888398870bc4c,21.5 Tỷ,66.20,66.2,2.0,1.0,certificate,30/10/2025,"801/43 Đường Xô Viết Nghệ Tĩnh, 25, Bình Thạnh..."
4,https://moso.vn/ban-6903475dd63888398870b555,1.8 Tỷ,26.00,9.3,1.0,1.0,certificate,30/10/2025,"40/13/27 Đường Số 2, 3, Gò Vấp, Hồ Chí Minh"


## 9) Chuẩn hoá cột số

In [9]:
df["Giá (VND)"]         = df["Giá"].apply(parse_price_to_vnd)
df["Diện tích sử dụng"] = df["Diện tích sử dụng"].apply(to_float)
df["Diện tích đất"]     = df["Diện tích đất"].apply(to_float)
df["Số ngày từ đăng"]   = df["Ngày đăng"].apply(parse_age_days)

cols_num = ["Giá (VND)","Diện tích sử dụng","Diện tích đất","Số ngày từ đăng"]
print("Sau chuẩn hoá số — xem 5 dòng:")
display(df[cols_num].head(5))


Sau chuẩn hoá số — xem 5 dòng:


,Giá (VND),Diện tích sử dụng,Diện tích đất,Số ngày từ đăng
0,2.350000e+10,256.51,162.0,1
1,4.800000e+09,NaN,79.8,1
2,4.800000e+09,60.60,77.1,1
3,2.150000e+10,66.20,66.2,1
4,1.800000e+09,26.00,9.3,1


## 10) Lọc ngoại lai IQR

In [10]:
for c in cols_num:
    df[c], (lo,hi), masked = iqr_mask(df[c], k=iqr_k)
    print(f"{c}: masked={masked}, clip≈[{fnum(lo)}, {fnum(hi)}]")

print("Sau IQR — xem 5 dòng:")
display(df[cols_num].head(5))


Giá (VND): masked=167, clip≈[-8500000000.0, 27500000000.0]
Diện tích sử dụng: masked=98, clip≈[-128.1, 409.4]
Diện tích đất: masked=185, clip≈[-38.8, 178.5]
Số ngày từ đăng: masked=0, clip≈[-90.0, 206.0]
Sau IQR — xem 5 dòng:


,Giá (VND),Diện tích sử dụng,Diện tích đất,Số ngày từ đăng
0,2.350000e+10,256.51,162.0,1
1,4.800000e+09,NaN,79.8,1
2,4.800000e+09,60.60,77.1,1
3,2.150000e+10,66.20,66.2,1
4,1.800000e+09,26.00,9.3,1


## 11) Điền giá trị thiếu 

In [11]:
if impute_method in {"median","mean"}:
    for c in cols_num:
        fill_val = df[c].median(skipna=True) if impute_method == "median" else df[c].mean(skipna=True)
        before = int(df[c].isna().sum())
        df[c] = df[c].fillna(fill_val)
        after = int(df[c].isna().sum())
        print(f"{c}: filled {before - after} NaN -> {impute_method}={fnum(fill_val)}")
else:
    print("Bỏ qua impute (impute_method='none').")

print("Sau impute — xem 5 dòng:")
display(df[cols_num].head(5))


Giá (VND): filled 167 NaN -> median=7100000000.0
Diện tích sử dụng: filled 558 NaN -> median=117.0
Diện tích đất: filled 272 NaN -> median=60.3
Số ngày từ đăng: filled 0 NaN -> median=51.0
Sau impute — xem 5 dòng:


,Giá (VND),Diện tích sử dụng,Diện tích đất,Số ngày từ đăng
0,2.350000e+10,256.51,162.0,1
1,4.800000e+09,117.05,79.8,1
2,4.800000e+09,60.60,77.1,1
3,2.150000e+10,66.20,66.2,1
4,1.800000e+09,26.00,9.3,1


## 12) Scale

In [12]:
for c in cols_num:
    df[f"{c}_std"] = scale_keep_nan(df[c], StandardScaler)
    df[f"{c}_mm"]  = scale_keep_nan(df[c], MinMaxScaler)

cols_scaled = [f"{c}_std" for c in cols_num] + [f"{c}_mm" for c in cols_num]
print("Sau scale — xem 10 dòng (các cột _std, _mm):")
display(df[cols_scaled].head(10))


Sau scale — xem 10 dòng (các cột _std, _mm):


,Giá (VND)_std,Diện tích sử dụng_std,Diện tích đất_std,Số ngày từ đăng_std,Giá (VND)_mm,Diện tích sử dụng_mm,Diện tích đất_mm,Số ngày từ đăng_mm
0,2.581706,1.652603,2.940006,-1.458052,0.849057,0.620116,0.911304,0.0
1,-0.717754,-0.218517,0.430876,-1.458052,0.143396,0.273511,0.434783,0.0
2,-0.717754,-0.975900,0.348460,-1.458052,0.143396,0.133214,0.419130,0.0
3,2.228822,-0.900766,0.015740,-1.458052,0.773585,0.147132,0.355942,0.0
4,-1.247079,-1.440125,-1.721114,-1.458052,0.030189,0.047221,0.026087,0.0
5,-0.100208,-0.372140,-1.019047,-1.458052,0.275472,0.245054,0.159420,0.0
6,-0.594245,-0.287614,-0.164355,-1.458052,0.169811,0.260712,0.321739,0.0
7,-0.064919,-0.134661,-0.610016,-1.458052,0.283019,0.289045,0.237101,0.0
8,0.640847,1.997282,1.657969,-1.458052,0.433962,0.683965,0.667826,0.0
9,0.640847,1.043342,0.284358,-1.458052,0.433962,0.507257,0.406957,0.0


## 13) Chuẩn hoá địa chỉ

In [13]:
parsed = df["Địa chỉ"].apply(split_hcm)
df["Tỉnh/TP"]         = parsed.apply(lambda x: x[0])
df["Quận/Huyện/TP"]   = parsed.apply(lambda x: x[1] or np.nan)
df["Phường/Xã/TT"]    = parsed.apply(lambda x: x[2] or np.nan)
df["Đường/Số nhà"]    = parsed.apply(lambda x: x[3] or np.nan)
df["Địa chỉ (chuẩn)"] = parsed.apply(lambda x: x[4] or np.nan)

cols_addr = ["Tỉnh/TP","Quận/Huyện/TP","Phường/Xã/TT","Đường/Số nhà","Địa chỉ (chuẩn)"]
print("Địa chỉ chuẩn — xem 10 dòng:")
display(df[cols_addr].head(10))


Địa chỉ chuẩn — xem 10 dòng:


,Tỉnh/TP,Quận/Huyện/TP,Phường/Xã/TT,Đường/Số nhà,Địa chỉ (chuẩn)
0,Hồ Chí Minh,TP Thủ Đức,Phường Hiệp Bình Chánh,"46 Đường Số 20, Hồ Chí Minh","46 Đường Số 20, Hiệp Bình Chánh, Thủ Đức, Hồ C..."
1,Hồ Chí Minh,Quận Gò Vấp,Phường 16,"Đường Lê Đức Thọ, Hồ Chí Minh","Đường Lê Đức Thọ, 16, Gò Vấp, Hồ Chí Minh"
2,Hồ Chí Minh,TP Thủ Đức,Phường Linh Xuân,"2/23A Đường số 13, Hồ Chí Minh","2/23A Đường số 13, Linh Xuân, Thủ Đức, Hồ Chí ..."
3,Hồ Chí Minh,Quận Bình Thạnh,Phường 25,"801/43 Đường Xô Viết Nghệ Tĩnh, Hồ Chí Minh","801/43 Đường Xô Viết Nghệ Tĩnh, 25, Bình Thạnh..."
4,Hồ Chí Minh,Quận Gò Vấp,Phường 3,"40/13/27 Đường Số 2, Hồ Chí Minh","40/13/27 Đường Số 2, 3, Gò Vấp, Hồ Chí Minh"
5,Hồ Chí Minh,Quận 3,Phường 13,"315/26A Đường Lê Văn Sỹ, Hồ Chí Minh","315/26A Đường Lê Văn Sỹ, 13, 3, Hồ Chí Minh"
6,Hồ Chí Minh,Huyện Bình Chánh,Xã Đa Phước,"D2/49 Quốc Lộ 50, Ấp 4, Hồ Chí Minh","D2/49 Quốc Lộ 50, Ấp 4, Đa Phước, Bình Chánh, ..."
7,Hồ Chí Minh,Quận 3,Phường 4,"524/16/19 Nguyễn Đình Chiểu, Hồ Chí Minh","524/16/19 Nguyễn Đình Chiểu, 4, 3, Hồ Chí Minh"
8,Hồ Chí Minh,Quận Bình Tân,Phường Tân Tạo A,"63 Đường Số 5, Khu nhà ở Bắc Lương Bèo, Hồ Chí...","63 Đường Số 5, Khu nhà ở Bắc Lương Bèo, Tân Tạ..."
9,Hồ Chí Minh,Quận 8,Phường 5,"152/29 Bông Sao, Hồ Chí Minh","152/29 Bông Sao, 5, 8, Hồ Chí Minh"


## 14) Top quận/phường

In [14]:
top_quan   = df["Quận/Huyện/TP"].dropna().astype(str).value_counts().head(5)
top_phuong = df["Phường/Xã/TT"].dropna().astype(str).value_counts().head(5)

print("Top 5 Quận/Huyện/TP:")
display(top_quan.to_frame("count").head(5))

print("Top 5 Phường/Xã/TT:")
display(top_phuong.to_frame("count").head(5))


Top 5 Quận/Huyện/TP:


,count
Quận/Huyện/TP,
Quận Gò Vấp,271
Quận Tân Bình,236
Quận Bình Tân,227
Quận Tân Phú,206
Quận 12,159


Top 5 Phường/Xã/TT:


,count
Phường/Xã/TT,
Phường 12,102
Phường 14,99
Phường 15,97
Phường 11,91
Phường 10,88


## 15) Lưu CSV

In [15]:
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu: {out_csv}")
print("Xem 5 dòng đầu của file kết quả:")
display(df.head(5))


✅ Đã lưu: moso_clean.csv
Xem 5 dòng đầu của file kết quả:


,URL,Giá,Diện tích sử dụng,Diện tích đất,Phòng ngủ,Phòng tắm,Giấy tờ pháp lý,Ngày đăng,Địa chỉ,Giá (VND),...,Diện tích sử dụng_mm,Diện tích đất_std,Diện tích đất_mm,Số ngày từ đăng_std,Số ngày từ đăng_mm,Tỉnh/TP,Quận/Huyện/TP,Phường/Xã/TT,Đường/Số nhà,Địa chỉ (chuẩn)
0,https://moso.vn/ban-690388d2d638883988715f20,23.5 Tỷ,256.51,162.0,4.0,4.0,certificate,30/10/2025,"46 Đường Số 20, Hiệp Bình Chánh, Thủ Đức, Hồ C...",2.350000e+10,...,0.620116,2.940006,0.911304,-1.458052,0.0,Hồ Chí Minh,TP Thủ Đức,Phường Hiệp Bình Chánh,"46 Đường Số 20, Hồ Chí Minh","46 Đường Số 20, Hiệp Bình Chánh, Thủ Đức, Hồ C..."
1,https://moso.vn/ban-690375a5d638883988712dd5,4.8 Tỷ,117.05,79.8,NaN,NaN,certificate,30/10/2025,"Đường Lê Đức Thọ, 16, Gò Vấp, Hồ Chí Minh",4.800000e+09,...,0.273511,0.430876,0.434783,-1.458052,0.0,Hồ Chí Minh,Quận Gò Vấp,Phường 16,"Đường Lê Đức Thọ, Hồ Chí Minh","Đường Lê Đức Thọ, 16, Gò Vấp, Hồ Chí Minh"
2,https://moso.vn/ban-69034b76d63888398870c2cf,4.8 Tỷ,60.60,77.1,6.0,6.0,certificate,30/10/2025,"2/23A Đường số 13, Linh Xuân, Thủ Đức, Hồ Chí ...",4.800000e+09,...,0.133214,0.348460,0.419130,-1.458052,0.0,Hồ Chí Minh,TP Thủ Đức,Phường Linh Xuân,"2/23A Đường số 13, Hồ Chí Minh","2/23A Đường số 13, Linh Xuân, Thủ Đức, Hồ Chí ..."
3,https://moso.vn/ban-69034961d63888398870bc4c,21.5 Tỷ,66.20,66.2,2.0,1.0,certificate,30/10/2025,"801/43 Đường Xô Viết Nghệ Tĩnh, 25, Bình Thạnh...",2.150000e+10,...,0.147132,0.015740,0.355942,-1.458052,0.0,Hồ Chí Minh,Quận Bình Thạnh,Phường 25,"801/43 Đường Xô Viết Nghệ Tĩnh, Hồ Chí Minh","801/43 Đường Xô Viết Nghệ Tĩnh, 25, Bình Thạnh..."
4,https://moso.vn/ban-6903475dd63888398870b555,1.8 Tỷ,26.00,9.3,1.0,1.0,certificate,30/10/2025,"40/13/27 Đường Số 2, 3, Gò Vấp, Hồ Chí Minh",1.800000e+09,...,0.047221,-1.721114,0.026087,-1.458052,0.0,Hồ Chí Minh,Quận Gò Vấp,Phường 3,"40/13/27 Đường Số 2, Hồ Chí Minh","40/13/27 Đường Số 2, 3, Gò Vấp, Hồ Chí Minh"
